# 03 - Auditoria de Data Quality e Engenharia Reversa
**Squad 2 — Real Time for Business | Dupla 1**  
**Integrantes:** Lucas Sousa Santos Oliveira & Zaiden Emiliano Segundo Seleme  
**Tabelas Auditadas:** `squad2.ecommerce_produtos` e `squad2.ecommerce_categorias`  

### Objetivo:
Atestar a qualidade e integridade dos dados lidos diretamente do Azure SQL Server após a execução da carga.
1. Estrutura (`printSchema`) e amostragem visual (`display`).
2. Contagem do volume de dados processados.
3. Auditoria de integridade nas chaves primárias (`sku` e `id_categoria`), confirmando status de 0 nulos.
4. Auditoria de integridade referencial entre produtos e categorias (`id_categoria`).

**Ordem no pipeline:** Executar após `02_carga_sqlserver.ipynb`.

In [0]:
import os
from dotenv import load_dotenv, find_dotenv
from pyspark.sql import functions as F

load_dotenv(find_dotenv(), override=True)

# Parâmetros de conexão com o Azure SQL Server
sql_host = os.getenv("SQL_HOST", "srv-database-intership.database.windows.net")
sql_database = os.getenv("SQL_DATABASE", "internshipDatabase")
sql_user = os.getenv("SQL_USERNAME", "estagiario_user")
sql_password = os.getenv("SQL_PASSWORD")

jdbc_url = f"jdbc:sqlserver://{sql_host}:1433;database={sql_database};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
jdbc_driver = "com.microsoft.sqlserver.jdbc.SQLServerDriver"

def ler_tabela_sql(schema_tabela):
    """Lê a tabela diretamente do Azure SQL Server utilizando o conector nativo sqlserver ou jdbc."""
    try:
        return spark.read \
            .format("sqlserver") \
            .option("host", sql_host) \
            .option("port", "1433") \
            .option("user", sql_user) \
            .option("password", sql_password) \
            .option("database", sql_database) \
            .option("dbtable", schema_tabela) \
            .load()
    except Exception:
        return spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("dbtable", schema_tabela) \
            .option("user", sql_user) \
            .option("password", sql_password) \
            .option("driver", jdbc_driver) \
            .load()

print("Módulo de auditoria de qualidade configurado com sucesso!")

## 1. Auditoria de Data Quality: `squad2.ecommerce_produtos`

In [0]:
# Leitura e Auditoria da tabela squad2.ecommerce_produtos
try:
    df_sql_produtos = ler_tabela_sql("squad2.ecommerce_produtos")
    print("=== Tabela 'squad2.ecommerce_produtos' encontrada no SQL Server ===")
    print("=== ESTRUTURA E ESQUEMA (printSchema) ===")
    df_sql_produtos.printSchema()
    
    total_linhas_prod = df_sql_produtos.count()
    total_colunas_prod = len(df_sql_produtos.columns)
    print(f"\nVolume Processado: {total_linhas_prod} linhas e {total_colunas_prod} colunas.")
    
    pk_col = "sku"
    nulos_pk = df_sql_produtos.filter(F.col(pk_col).isNull()).count()
    distintos_pk = df_sql_produtos.select(pk_col).distinct().count()
    duplicados_pk = total_linhas_prod - distintos_pk
    
    print(f"\n--- Auditoria de Integridade na Chave Primária ({pk_col}) ---")
    print(f"Registros Nulos: {nulos_pk} (Status: {'[SAUDÁVEL - 0 NULOS]' if nulos_pk == 0 else '[ALERTA: Nulos Detectados]'})")
    print(f"Registros Duplicados na PK: {duplicados_pk}")
    
    print("\n--- Nulos por Coluna ---")
    df_sql_produtos.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_sql_produtos.columns if not c.startswith("_")]).show()
    
    print("\n--- Amostragem Visual dos Dados Carregados (display) ---")
    display(df_sql_produtos.limit(10))

except Exception as e:
    if "Invalid object name" in str(e):
        print("""
[!] AVISO: A tabela 'squad2.ecommerce_produtos' ainda NÃO foi criada no Azure SQL Server.
-> Para resolver: Execute primeiro o notebook '03_carga_sqlserver.ipynb' para realizar a gravação dos dados no banco.
-> Em seguida, execute este notebook novamente para atestar a qualidade dos dados carregados.
""")
    else:
        print(f"Erro ao conectar com SQL Server: {e}")

## 2. Auditoria de Data Quality: `squad2.ecommerce_categorias`

In [0]:
# Leitura e Auditoria da tabela squad2.ecommerce_categorias
try:
    df_sql_categorias = ler_tabela_sql("squad2.ecommerce_categorias")
    print("=== Tabela 'squad2.ecommerce_categorias' encontrada no SQL Server ===")
    print("=== ESTRUTURA E ESQUEMA (printSchema) ===")
    df_sql_categorias.printSchema()
    
    total_linhas_cat = df_sql_categorias.count()
    total_colunas_cat = len(df_sql_categorias.columns)
    print(f"\nVolume Processado: {total_linhas_cat} linhas e {total_colunas_cat} colunas.")
    
    pk_col_cat = "id_categoria"
    nulos_pk_cat = df_sql_categorias.filter(F.col(pk_col_cat).isNull()).count()
    distintos_pk_cat = df_sql_categorias.select(pk_col_cat).distinct().count()
    duplicados_pk_cat = total_linhas_cat - distintos_pk_cat
    
    print(f"\n--- Auditoria de Integridade na Chave Primária ({pk_col_cat}) ---")
    print(f"Registros Nulos: {nulos_pk_cat} (Status: {'[SAUDÁVEL - 0 NULOS]' if nulos_pk_cat == 0 else '[ALERTA: Nulos Detectados]'})")
    print(f"Registros Duplicados na PK: {duplicados_pk_cat}")
    
    print("\n--- Nulos por Coluna ---")
    df_sql_categorias.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_sql_categorias.columns if not c.startswith("_")]).show()
    
    print("\n--- Amostragem Visual dos Dados Carregados (display) ---")
    display(df_sql_categorias.limit(10))

except Exception as e:
    if "Invalid object name" in str(e):
        print("""
[!] AVISO: A tabela 'squad2.ecommerce_categorias' ainda NÃO foi criada no Azure SQL Server.
-> Para resolver: Execute primeiro o notebook '03_carga_sqlserver.ipynb' para realizar a gravação dos dados no banco.
-> Em seguida, execute este notebook novamente para atestar a qualidade dos dados carregados.
""")
    else:
        print(f"Erro ao conectar com SQL Server: {e}")

## 3. Auditoria de Integridade Referencial (Produtos x Categorias)

In [0]:
try:
    if 'df_sql_produtos' in locals() and 'df_sql_categorias' in locals():
        fk = "id_categoria"
        print(f"Checando integridade referencial através da chave estrangeira '{fk}'...")
        
        prod_c = df_sql_produtos.withColumn("fk_cat", F.col(fk).cast("string"))
        cat_c = df_sql_categorias.withColumn("fk_cat", F.col(fk).cast("string"))
        
        orfaos = prod_c.join(cat_c, on="fk_cat", how="left_anti").count()
        print(f"Produtos com categoria inexistente (órfãos): {orfaos} (Status: {'[SAUDÁVEL - 0 ÓRFÃOS]' if orfaos == 0 else f'[ATENÇÃO: {orfaos} Órfãos]'})")
    else:
        print("Auditoria de relacionamento aguardando a carga de ambas as tabelas no SQL Server.")
except Exception as e:
    print(f"Aviso na integridade referencial: {e}")